# Figures behind the `[viz]` extra

`axiom.viz` never imports plotly at import time. `available()` says whether the extra is
installed; every figure function returns a plotly `Figure` when it is and a typed
`Unsupported(reason="plotly not installed", missing=("viz",))` when it is not. Figures are
duck-typed over the result objects — each reads a documented set of attributes, so a missing
attribute is also a typed failure. Labels use the general vocabulary (treatment, dose,
outcome). Here we print figure sizes rather than rendering.

In [ ]:
import numpy as np

from axiom.core import Unsupported, clopper_pearson
from axiom.diagnose import (
    SpecificationAxis, SpecOption, coverage, rank_uniformity, rolling_origin, specification_curve,
)
from axiom.meta import forest_data, funnel_data, random_effects
from axiom.sim import DosePlan, surface_world
from axiom.surface import GeometricCarryover, fit
from axiom.viz import (
    available, backtest_plot, coverage_plot, forest, funnel, marginal_curve, response_curve,
    sbc_ranks, spec_curve_plot,
)

print("plotly available:", available())

## Response curve, forest, and funnel

`response_curve(result, treatment)` evaluates the fitted surface on a dose grid through
`predict_under` (one forward per draw) and draws the mean with a posterior band; `forest` and
`funnel` take `meta.forest_data` / `meta.funnel_data`.

In [ ]:
world = surface_world(n_units=2, n_periods=6, treatments=("a",), doses=DosePlan(scale=10.0), intercept="shared", seed=3)
res = fit(world.spec, world.panel, backend="laplace", draws=40, chains=1, seed=4)
fig = response_curve(res, "a", n_grid=8, mass=0.8)
print("response_curve traces:", len(fig.data), "| x:", fig.layout.xaxis.title.text, "| y:", fig.layout.yaxis.title.text)
# both surface figures render a surface.ResponseBand, so the band is the first trace and
# there is no argument on either that turns it off (tests/contracts/test_surface_uncertainty.py)
slope = marginal_curve(res, "a", n_grid=8, mass=0.8)
print("marginal_curve traces:", len(slope.data), "| band first:", slope.data[0].fill == "toself")

y, se = np.array([0.42, 0.55, 0.31, 0.67, 0.48]), np.array([0.10, 0.15, 0.12, 0.20, 0.11])
pooled = random_effects(y, se)
print("forest traces:", len(forest(forest_data(y, se, pooled, labels=list("abcde"))).data))
print("funnel traces:", len(funnel(funnel_data(y, se, pooled)).data))

## Diagnostic figures

`sbc_ranks` draws one histogram per `ParameterRanks`; `coverage_plot` shows each parameter's
rate against its acceptance region; `spec_curve_plot` sorts the rows of a `SpecCurve`;
`backtest_plot` shows the per-horizon scores of a `Backtest`. Each accepts any object with the
documented attributes — here real results from `axiom.diagnose` on tiny worlds.

In [ ]:
from types import SimpleNamespace

rng = np.random.default_rng(0)
sbc_like = SimpleNamespace(parameters=(
    rank_uniformity("beta_a", rng.integers(0, 20, size=60), n_ranks=19, alpha=0.05),
    rank_uniformity("k_a", np.minimum(rng.integers(0, 8, size=60), 19), n_ranks=19, alpha=0.05),
))
print("sbc_ranks traces:", len(sbc_ranks(sbc_like, columns=2).data))


def make_world(seed: int):
    return surface_world(n_units=3, n_periods=6, treatments=("a",), intercept="shared", noise_sd=0.3, seed=seed, doses=DosePlan(zero_fraction=0.2))


cov = coverage(make_world, n=4, draws=40, seed=1)
print("coverage_plot traces:", len(coverage_plot(cov).data))

In [ ]:
small = surface_world(n_units=3, n_periods=8, treatments=("a",), intercept="shared", doses=DosePlan(scale=50.0, zero_fraction=0.2), noise_sd=0.2, seed=11)
axes = (SpecificationAxis(name="intercept_scale", options=(SpecOption(label="tight", spec_update={"intercept_scale": 0.5}), SpecOption(label="wide", spec_update={"intercept_scale": 5.0}))),)
curve = specification_curve(small.spec, small.panel, axes, draws=30, chains=1, seed=3)
print("spec_curve_plot traces:", len(spec_curve_plot(curve).data))

carry = surface_world(n_units=3, n_periods=10, treatments=("a",), carryover={"a": GeometricCarryover(max_lag=3)}, intercept="shared", doses=DosePlan(scale=50.0, zero_fraction=0.2), noise_sd=0.1, seed=31)
bt = rolling_origin(carry.spec, carry.panel, origins=(7,), horizon=2, draws=30, chains=1, seed=0)
fig = backtest_plot(bt)
print("backtest_plot traces:", len(fig.data), "| json head:", fig.to_json()[:80])

## Typed failures

A result missing a documented attribute gives `Unsupported` naming what is missing; the same
happens for every figure when plotly is absent (the unit tests monkeypatch the import to
assert that path).

In [ ]:
out = coverage_plot(SimpleNamespace(mass=0.9, parameters=(SimpleNamespace(name="x"),)))
assert isinstance(out, Unsupported)
print(out.reason, "| missing:", out.missing)
print(backtest_plot(SimpleNamespace(mass=0.9)).missing)

## The structure itself

Two figures about *structure* rather than about an estimate. A causal package
whose central object had no picture is asking a reader to hold a graph in their
head while they read a verdict about it.

In [ ]:
from axiom.identify import CausalGraph
from axiom.viz import causal_graph

graph = CausalGraph.from_edges(
    "age -> dose, age -> pressure, dose -> adherence, adherence -> pressure, "
    "u -> dose, u -> pressure",
    unmeasured=["u"],
    name="HYPER-3",
)
figure = causal_graph(graph)
print(type(figure).__name__, "with", len(figure.layout.annotations), "arrows")
figure

Depth is the longest path from a root, so every arrow points forwards and none
doubles back. Unmeasured nodes are drawn hollow and bidirected edges dashed,
because those two are exactly what separate a graph you can identify from one
you cannot.

## What a resample settles, and what it does not

In [ ]:
import numpy as np

from axiom.discover import Dataset, edge_stability
from axiom.viz import stability

rng = np.random.default_rng(0)
n = 400
a = rng.normal(size=n)
b = 1.4 * a + rng.normal(size=n)
d = 0.8 * a + rng.normal(size=n)
c = -0.9 * b + 1.1 * d + rng.normal(size=n)

data = Dataset.observational(np.column_stack([a, b, c, d]), ["a", "b", "c", "d"])
report = edge_stability(data, n_bootstrap=25, seed=1)
stability(report)

The split is the finding. A bar that is nearly all *undirected* is a stable edge
whose direction observation cannot settle — more rows will not move it, and only
an intervention will. A short bar is an edge the data is unsure about at all.
One number would lose the distinction that decides what to do next.

## One figure per subpackage

Every subpackage that returns a result now has a picture of it. The ones that do
not — `io`, `build`, `adapters`, `report`, `display` — return plumbing rather
than findings, and a chart of a file format would be decoration.

### `infer` — did the sampler give you something usable?

In [ ]:
import numpy as np

from axiom.core import Posterior
from axiom.infer import diagnose
from axiom.viz import convergence

rng = np.random.default_rng(0)
posterior = Posterior(
    {"alpha": rng.normal(size=(4, 600)), "beta": rng.normal(size=(4, 600))}
)
report = diagnose(posterior)
convergence(report)

The question a sampler's output poses is "may I use this?", and the answer is
per parameter — one bad row is enough. A row with no R-hat is drawn at zero and
named, because *not computed* is a different state from *fine*.

### `design` — the stopping rule, which is the protocol sentence

In [ ]:
from axiom.design import obrien_fleming
from axiom.viz import boundary

rule = obrien_fleming(0.025, [0.25, 0.5, 0.75, 1.0], kind="efficacy")
boundary(rule)

Drawn as a step, because the threshold holds *until* the next look rather than
sliding between them. The alpha spent by each look is in the hover rather than
on a second axis — two scales on one plot is the chart mistake this package
refuses everywhere else.

### `calibrate` — which correction moved the number

In [ ]:
from axiom.calibrate import variance_reweight
from axiom.viz import corrections

applied = [
    variance_reweight(0.40, n_source=1200, n_target=300),
    variance_reweight(0.31, n_source=300, n_target=900, icc=0.05, cluster_size=20),
]
corrections(applied)

A calibrated estimate is a raw one with operators applied, and the useful
question is never "what is the answer" but "which correction moved it". The
single corrected number cannot answer that; a waterfall can.

### `data` — the shape everything downstream depends on

In [ ]:
import pandas as pd

from axiom.core import Outcome
from axiom.data import Panel, RoleMap
from axiom.viz import panel_coverage

frame = pd.DataFrame(
    {
        "unit": ["a"] * 5 + ["b"] * 4 + ["c"] * 5,
        "period": [1, 2, 3, 4, 5, 1, 2, 4, 5, 1, 2, 3, 4, 5],
        "y": rng.normal(size=14),
    }
)
panel = Panel(frame, RoleMap(unit="unit", time="period", outcome=("y", Outcome(name="y"))))
panel_coverage(panel)

Unbalance has a *pattern*, and the pattern says whether units dropped out,
arrived late, or were never there. Unit `b` is missing period 3 — a table of
counts would report the gap and not where it is.

### `core` — several intervals on one scale

In [ ]:
from axiom.core import Interval
from axiom.viz import intervals

intervals(
    {
        "40 mg": Interval(lower=-16.7, upper=-8.1, definition="eti", mass=0.9),
        "20 mg": Interval(lower=-6.2, upper=-3.4, definition="eti", mass=0.9),
        "10 mg": Interval(lower=-2.1, upper=1.4, definition="eti", mass=0.9),
    },
    unit="mmHg",
)

The comparison a reader makes by hand, made once — overlap is only visible on a
shared scale.

### `sim` — did the estimator find what was there?

In [ ]:
from axiom.viz import recovery

truth = {"beta": 2.0, "gamma": -0.75, "sigma": 1.0}
estimated = {"beta": 1.94, "gamma": -0.81, "sigma": 1.06}
recovery(truth, estimated)

The picture every recovery test is implicitly making. Distance from the diagonal
is bias, visible at a glance in a way a table of differences is not.

### `dynamics` — feedback becomes a DAG once time is explicit

In [ ]:
from axiom.core import D, Param, dimensionless
from axiom.dynamics import Variable, parse_system, unroll
from axiom.viz import unrolled

system = parse_system(
    "stock = decay * stock[t-1] + beta * inflow",
    variables=(
        Variable(name="stock", dimension=D.outcome, initial=0.0),
        Variable(name="inflow", dimension=D.currency, role="exogenous"),
    ),
    parameters=(
        Param(name="decay", dimension=dimensionless()),
        Param(name="beta", dimension=D.outcome / D.currency),
    ),
    name="one-compartment",
)
unrolled(unroll(system, periods=4))

The layered layout puts each node one step right of its parents, so the
horizontal axis *is* the period — no parsing of column names, and a cycle that
survived the unroll would show as a node pushed to the far right rather than
hidden.

### `estimands` — what it costs to move a result

In [ ]:
from axiom.core import D, Intervention, Outcome, Population, TimeWindow, Treatment
from axiom.estimands import Estimand, Level, Quantity
from axiom.viz import transfer

fertilizer = Treatment(name="fertilizer", dimension=D.currency, unit="USD")
base = dict(
    quantity=Quantity(kind="contrast"),
    treatment=fertilizer,
    intervention=Intervention(doses={"fertilizer": 100.0}, version="granular"),
    reference=Intervention(doses={"fertilizer": 0.0}, version="granular"),
    outcome=Outcome(name="yield_total", dimension=D.outcome, unit="kg"),
    window=TimeWindow(start=0, stop=8),
    level=Level(unit="cluster"),
    dimension=D.outcome,
)
trial = Estimand(
    name="experiment_lift",
    population=Population(name="north", strata={"soil": {"clay": 0.3, "loam": 0.7}}),
    **base,
)
region = Estimand(
    name="region_lift",
    population=Population(name="whole_region", strata={"soil": {"clay": 0.6, "loam": 0.4}}),
    **base,
)

plan = trial.transfer_to(region)
print(plan.status, "| differing facets:", plan.differing)
transfer(plan)


A transfer plan's finding is a set: these facets differ, so these corrections are
required and these assumptions come with them. The count of differing facets is
what decides whether a result travels at all.